### [DENEY 1]
**Amaç:** Referans makaledeki baz mimariyi (M1) yeniden üretmek ve önerdiğimiz M2 mimarisi ile doğrudan karşılaştırmak.
**Bu Hücrede Değişen Parametreler / Yapılan Ayarlar:**
- **ELA Ölçek Faktörü (`scale`):** 15 (Mikro sınır anomalilerini kuvvetlendirmek için yükseltildi).
- **Veri Kümesi Bölütleme (Split):** %80 Eğitim, %20 Test (Doğrudan `train_test_split` ile ayrıldı, validasyon kanalı yok).
- **Model Yapıları:** - *M1:* Orijinal 2 evrişimli katman (32 filtre, $5\times5$ kernel), 150 nöronlu Dense katmanı.
  - *M2:* Batch Normalization katmanları, ek 2 adet evrişim katmanı (64 filtre, $5\times5$ kernel) ve 256 nöronlu Dense katmanı eklendi.
- **Epoch Sayısı:** 40.

In [ ]:
import os
import numpy as np
from PIL import Image, ImageChops, ImageEnhance
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

os.environ['KAGGLE_API_TOKEN'] = 'KGAT_0c19a9db6740e7da1d8da96086f765c2'
!pip install kaggle -q
!kaggle datasets download -d sophatvathana/casia-dataset -q
!unzip -q -o casia-dataset.zip -d casia
print("Veri hazır!")

In [ ]:
def ela(image_path, quality=90, scale=15):
    try:
        original = Image.open(image_path).convert('RGB')
        original.save('/tmp/temp_ela.jpg', 'JPEG', quality=quality)
        compressed = Image.open('/tmp/temp_ela.jpg')
        diff = ImageChops.difference(original, compressed)
        extrema = diff.getextrema()
        max_diff = max([ex[1] for ex in extrema]) or 1
        diff = ImageEnhance.Brightness(diff).enhance(255.0 / max_diff * scale)
        return np.array(diff.resize((150, 150))) / 255.0
    except:
        return np.zeros((150, 150, 3))

def get_jpeg_files(folder):
    return [os.path.join(folder, f) for f in os.listdir(folder)
            if f.lower().endswith(('.jpg', '.jpeg'))]

au_files = get_jpeg_files('casia/CASIA2/Au')
tp_files = get_jpeg_files('casia/CASIA2/Tp')
all_files = au_files + tp_files
all_labels = [0]*len(au_files) + [1]*len(tp_files)

# 80-20 split (makaledeki gibi)
X_train_f, X_test_f, y_train, y_test = train_test_split(
    all_files, all_labels, test_size=0.2, stratify=all_labels, random_state=42)

print(f"Train: {len(X_train_f)}, Test: {len(X_test_f)}")

In [ ]:
def data_generator(file_list, label_list, batch_size=8):
    while True:
        idx = np.random.permutation(len(file_list))
        for start in range(0, len(file_list), batch_size):
            batch_idx = idx[start:start+batch_size]
            X = np.array([ela(file_list[i]) for i in batch_idx])
            y = np.array([label_list[i] for i in batch_idx])
            # 2 sınıf için one-hot encoding
            y_onehot = tf.keras.utils.to_categorical(y, num_classes=2)
            yield X, y_onehot

steps_train = len(X_train_f) // 8
steps_test  = len(X_test_f) // 8

train_gen = data_generator(X_train_f, y_train)
test_gen  = data_generator(X_test_f, y_test)
print("Generator hazır!")

In [ ]:
model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    layers.Conv2D(32, (5,5), activation='relu'),
    layers.Conv2D(32, (5,5), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(150, activation='relu'),
    layers.Dropout(0.25),
    layers.Dense(2, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train_gen,
    steps_per_epoch=steps_train,
    epochs=40,
    callbacks=[
        ModelCheckpoint('best_model_reproduce.keras', save_best_only=True, monitor='accuracy')
    ]
)
print("Reproduce tamamlandı!")


In [ ]:
y_pred, y_true = [], []
for i in range(0, len(X_test_f), 8):
    X_batch = np.array([ela(f) for f in X_test_f[i:i+8]])
    preds = model.predict(X_batch, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(y_test[i:i+8])

print("=== REPRODUCE SONUÇLARI ===")
print(classification_report(y_true, y_pred, target_names=['Orijinal', 'Sahte']))

with open('/kaggle/working/reproduce_results.txt', 'w') as f:
    f.write(classification_report(y_true, y_pred, target_names=['Orijinal', 'Sahte']))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='Train')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'], label='Train')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/reproduce_grafik.png', dpi=150)
plt.show()


In [ ]:
model2 = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    layers.Conv2D(32, (5,5), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (5,5), activation='relu', padding='same'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    layers.Conv2D(64, (5,5), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (5,5), activation='relu', padding='same'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(2, activation='sigmoid')
])

model2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history2 = model2.fit(
    train_gen,
    steps_per_epoch=steps_train,
    epochs=40,
    callbacks=[
        ModelCheckpoint('best_model2.keras', save_best_only=True, monitor='accuracy')
    ]
)
print("Model 2 tamamlandı!")

In [ ]:
def test_model(mdl, name):
    y_pred, y_true = [], []
    for i in range(0, len(X_test_f), 8):
        X_batch = np.array([ela(f) for f in X_test_f[i:i+8]])
        preds = mdl.predict(X_batch, verbose=0)
        y_pred.extend(np.argmax(preds, axis=1))
        y_true.extend(y_test[i:i+8])
    report = classification_report(y_true, y_pred, target_names=['Orijinal', 'Sahte'])
    print(f"\n=== {name} ===")
    print(report)
    with open(f'/kaggle/working/{name}.txt', 'w') as f:
        f.write(report)

test_model(model,  "Model1_Reproduce")
test_model(model2, "Model2_Iyilestirme")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(history.history['accuracy'], label='Train')
axes[0,0].set_title('Model 1 - Accuracy'); axes[0,0].legend()

axes[0,1].plot(history.history['loss'], label='Train')
axes[0,1].set_title('Model 1 - Loss'); axes[0,1].legend()

axes[1,0].plot(history2.history['accuracy'], label='Train')
axes[1,0].set_title('Model 2 - Accuracy'); axes[1,0].legend()

axes[1,1].plot(history2.history['loss'], label='Train')
axes[1,1].set_title('Model 2 - Loss'); axes[1,1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/tum_grafikler.png', dpi=150)
plt.show()
print("Her şey tamamlandı!")